In [3]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict
import torch
import re 
import pandas as pd

# Load pre-trained tokenizer and model
model_name = "neuralmind/bert-large-portuguese-cased"  # BERTimbau Base
tokenizer = BertTokenizer.from_pretrained(model_name)

# Define labels (11 categories from TuPy-E)
hate_labels = ['ageism', 'aporophobia', 'body_shame', 'capacitism', 'lgbtphobia',
               'political', 'racism', 'religious_intolerance', 'misogyny', 'xenophobia', 'other']


In [4]:
from datasets import load_dataset

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = [float(example[label]) for label in hate_labels]
    return enc

tokenized_ds = ds.map(preprocess)


Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [14]:
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=len(hate_labels), problem_type="multi_label_classification")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model checkpoints
    eval_strategy="epoch",  # Evaluate after each epoch
    
    logging_strategy="no",  # Disables logging
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=10,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='eval_subset_accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from torch.utils.data import default_collate
import torch

class MultilabelDataCollator:
    def __call__(self, features):
        # Convert everything to tensors
        batch = {
            key: torch.tensor([f[key] for f in features])
            for key in features[0]
        }
        return batch

collator = MultilabelDataCollator()

from sklearn.metrics import f1_score, accuracy_score, classification_report, precision_score, recall_score

def compute_metrics(pred):
    logits, labels = pred.predictions, pred.label_ids
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    pred_labels = (probs >= 0.5).astype(int)

    # Subset accuracy (exact match across all labels)
    subset_acc = accuracy_score(labels, pred_labels)

    # Micro F1 (better for imbalanced multilabel data)
    micro_f1 = f1_score(labels, pred_labels, average='micro', zero_division=0)
    micro_precision = precision_score(labels, pred_labels, average='micro', zero_division=0)
    micro_recall = recall_score(labels, pred_labels, average='micro', zero_division=0)

    return {
        'subset_accuracy': subset_acc,
        'micro_f1': micro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=MultilabelDataCollator(),
    compute_metrics=compute_metrics  # ← This is the key addition
)

/tmp/ipykernel_31/1073832255.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro F1,Micro Precision,Micro Recall
1,No log,0.048820,0.842798,0.473597,0.616249,0.384574
2,No log,0.045172,0.855965,0.527831,0.648585,0.444984
3,No log,0.046442,0.846119,0.584748,0.584276,0.585221
4,No log,0.050962,0.843829,0.557034,0.586063,0.530744
5,No log,0.058067,0.851385,0.551621,0.621786,0.495685
6,No log,0.064917,0.849324,0.552879,0.607097,0.507551
7,No log,0.070781,0.854362,0.544499,0.637482,0.475189
8,No log,0.071895,0.849553,0.552616,0.607235,0.507012
9,No log,0.073356,0.851042,0.556378,0.616393,0.507012
10,No log,0.074877,0.851614,0.561362,0.615979,0.515642


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=10920, training_loss=0.02034760499611879, metrics={'train_runtime': 17974.4798, 'train_samples_per_second': 19.435, 'train_steps_per_second': 0.608, 'total_flos': 8.139270062028288e+16, 'train_loss': 0.02034760499611879, 'epoch': 10.0})

In [22]:
from sklearn.metrics import classification_report
import numpy as np
import torch

# Predict on test set
predictions = trainer.predict(tokenized_ds["test"])
logits = predictions.predictions
probs = torch.sigmoid(torch.tensor(logits)).numpy()
pred_labels = (probs >= 0.5).astype(int)

# Add non_hate column: 1 if no hate labels predicted
non_hate_preds = (pred_labels.sum(axis=1) == 0).astype(int)
non_hate_true = (predictions.label_ids.sum(axis=1) == 0).astype(int)

# Append non_hate as the 12th label
pred_labels = np.concatenate([pred_labels, non_hate_preds[:, None]], axis=1)
true_labels = np.concatenate([predictions.label_ids, non_hate_true[:, None]], axis=1)

# Updated label names
full_labels = hate_labels + ["non_hate"]

# Classification report
print(classification_report(true_labels, pred_labels, target_names=full_labels, zero_division=0))


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


                       precision    recall  f1-score   support

               ageism       0.00      0.00      0.00        12
          aporophobia       0.00      0.00      0.00        14
           body_shame       0.82      0.52      0.64        63
           capacitism       0.00      0.00      0.00        11
           lgbtphobia       0.80      0.63      0.70       149
            political       0.78      0.24      0.37       230
               racism       0.50      0.07      0.12        56
religious_intolerance       0.00      0.00      0.00        17
             misogyny       0.66      0.64      0.65       335
           xenophobia       0.53      0.09      0.16        88
                other       0.61      0.47      0.53       879
             non_hate       0.90      0.96      0.93      7188

            micro avg       0.86      0.85      0.86      9042
            macro avg       0.47      0.30      0.34      9042
         weighted avg       0.84      0.85      0.84 

In [27]:
trainer.save_model("./my-bertimbau-hate-large-model-cat")
tokenizer.save_pretrained("./my-bertimbau-hate-large-model-cat")


('./my-bertimbau-hate-large-model-cat/tokenizer_config.json',
 './my-bertimbau-hate-large-model-cat/special_tokens_map.json',
 './my-bertimbau-hate-large-model-cat/vocab.txt',
 './my-bertimbau-hate-large-model-cat/added_tokens.json')